In [ ]:
import os
import librosa
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

DATASET_PATH = "../Indian_Languages_Audio_Dataset"

def extract_mfcc(file_path, n_mfcc=13):
    audio, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfcc.T, axis=0)

X, y = [], []

for language in os.listdir(DATASET_PATH):
    lang_path = os.path.join(DATASET_PATH, language)
    if not os.path.isdir(lang_path):
        continue

    for file in os.listdir(lang_path):
        if file.endswith(".mp3"):
            path = os.path.join(lang_path, file)
            X.append(extract_mfcc(path))
            y.append(language)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

audio_model = RandomForestClassifier(n_estimators=200)
audio_model.fit(X_train, y_train)

preds = audio_model.predict(X_test)

print("Audio Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

with open("../model/audio_language_model.pkl", "wb") as f:
    pickle.dump(audio_model, f)

print("Audio language model saved")


Audio Accuracy: 0.811
              precision    recall  f1-score   support

     Bengali       0.82      0.91      0.86       175
    Gujarati       0.48      0.42      0.45       208
       Hindi       0.90      0.94      0.92       201
     Kannada       0.97      0.90      0.93       211
   Malayalam       0.92      0.93      0.93       192
     Marathi       0.92      0.88      0.90       212
     Punjabi       0.44      0.45      0.44       202
       Tamil       0.90      0.97      0.93       204
      Telugu       0.89      0.89      0.89       186
        Urdu       0.84      0.86      0.85       209

    accuracy                           0.81      2000
   macro avg       0.81      0.81      0.81      2000
weighted avg       0.81      0.81      0.81      2000

✅ Audio language model saved
